# 02 — Exploratory Data Analysis
Parking trends by time, weather impact, neighborhood comparisons, and event effects.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA = Path('../data')
VIZ  = Path('../visualizations')
sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
live     = pd.read_csv(DATA / 'live_parking_occupancy.csv', parse_dates=['occupancy_date'])
weather  = pd.read_csv(DATA / 'processed_weather_data.csv', parse_dates=['timestamp'])
merged   = pd.read_csv(DATA / 'merged_parking_weather_data.csv', parse_dates=['Date Time'])
events   = pd.read_csv(DATA / 'seattle_events.csv', parse_dates=['event_date'])
holidays = pd.read_csv(DATA / 'seattle_holidays.csv', parse_dates=['date'])

## 1. Occupancy by Hour of Day

In [ ]:
live['hour_of_day'] = live['occupancy_hour'].astype(int)
hourly = live.groupby('hour_of_day')['occupancy_rate'].mean()

plt.figure(figsize=(12, 4))
plt.bar(hourly.index, hourly.values, color='steelblue', alpha=0.8)
plt.axvspan(7, 9, alpha=0.15, color='red', label='AM Peak')
plt.axvspan(16, 19, alpha=0.15, color='orange', label='PM Peak')
plt.title('Average Parking Occupancy Rate by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Occupancy Rate')
plt.xticks(range(0, 24))
plt.legend()
plt.tight_layout()
plt.savefig(VIZ / 'parking/occupancy_by_hour.png', dpi=150)
plt.show()

## 2. Weekday vs Weekend

In [ ]:
live['day_of_week'] = live['occupancy_date'].dt.dayofweek
live['day_name']    = live['occupancy_date'].dt.day_name()
live['is_weekend']  = live['day_of_week'] >= 5

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
by_day = live.groupby('day_name')['occupancy_rate'].mean().reindex(day_order)

colors = ['#2196F3']*5 + ['#FF9800']*2
plt.figure(figsize=(10, 4))
plt.bar(by_day.index, by_day.values, color=colors)
plt.title('Average Occupancy Rate by Day of Week')
plt.ylabel('Occupancy Rate')
plt.tight_layout()
plt.savefig(VIZ / 'parking/occupancy_by_day.png', dpi=150)
plt.show()

## 3. Weather Impact on Parking

In [ ]:
merged['hour'] = merged['Date Time'].dt.hour
numeric_cols = ['Parking_Spaces', 'Total_Vehicle_Count', 'temperature', 'precipitation', 'wind_speed']
corr = merged[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation: Parking vs Weather Variables')
plt.tight_layout()
plt.savefig(VIZ / 'merged/weather_parking_correlation.png', dpi=150)
plt.show()

In [ ]:
merged['is_raining'] = merged['precipitation'] > 0.5
rain_effect = merged.groupby('is_raining')['Total_Vehicle_Count'].mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, label, color in zip(
    axes,
    ['temperature', 'precipitation', 'wind_speed'],
    ['Temperature (°C)', 'Precipitation (mm)', 'Wind Speed (m/s)'],
    ['tomato', 'steelblue', 'seagreen']
):
    ax.scatter(merged[col], merged['Total_Vehicle_Count'], alpha=0.05, s=1, color=color)
    ax.set_xlabel(label)
    ax.set_ylabel('Vehicle Count')
    ax.set_title(f'Vehicle Count vs {label}')
plt.tight_layout()
plt.savefig(VIZ / 'merged/weather_vs_vehicles.png', dpi=150)
plt.show()

## 4. Occupancy by Neighborhood (Annual Study)

In [ ]:
top_areas = (
    merged.groupby('Study_Area')['Total_Vehicle_Count']
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 5))
top_areas.plot(kind='bar', color='steelblue', alpha=0.85)
plt.title('Top 10 Areas by Average Vehicle Count')
plt.ylabel('Average Vehicle Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(VIZ / 'parking/top_areas_by_demand.png', dpi=150)
plt.show()

## 5. Event Impact on Occupancy

In [ ]:
live['date'] = live['occupancy_date'].dt.date
event_dates  = set(events['event_date'].dt.date)
live['is_event_day'] = live['date'].isin(event_dates)

event_compare = live.groupby('is_event_day')['occupancy_rate'].mean()
labels = ['No Event', 'Event Day']
plt.figure(figsize=(6, 4))
plt.bar(labels, event_compare.values, color=['#90CAF9', '#EF5350'])
plt.title('Average Occupancy: Event vs Non-Event Days')
plt.ylabel('Occupancy Rate')
plt.ylim(0, 1)
for i, v in enumerate(event_compare.values):
    plt.text(i, v + 0.01, f'{v:.2%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ / 'parking/event_vs_nonevent_occupancy.png', dpi=150)
plt.show()

## 6. Holiday Impact

In [ ]:
holiday_dates = set(holidays['date'].dt.date)
live['is_holiday'] = live['date'].isin(holiday_dates)

hol_compare = live.groupby('is_holiday')['occupancy_rate'].mean()
labels = ['Regular Day', 'Holiday']
plt.figure(figsize=(6, 4))
plt.bar(labels, hol_compare.values, color=['#90CAF9', '#CE93D8'])
plt.title('Average Occupancy: Holiday vs Regular Days')
plt.ylabel('Occupancy Rate')
plt.ylim(0, 1)
for i, v in enumerate(hol_compare.values):
    plt.text(i, v + 0.01, f'{v:.2%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(VIZ / 'parking/holiday_vs_regular_occupancy.png', dpi=150)
plt.show()